# Lab 21 — Fine-tuning LLMs · RUN ALL (Kaggle, T4)

Trước khi chạy: Settings (panel phải) -> **Accelerator: GPU T4 x2**, **Internet: On** (cần verify số điện thoại).

| Ô | Làm gì | Thời gian |
|---|---|---|
| 1 | clone + install | ~1-2 phút |
| 2 | smoke: import + unit test | ~30 giây |
| 3 | **core pipeline NB1 -> NB5** | ~100-130 phút |
| 4 | gatekeeper + in kết quả | ~10 giây |

Quota GPU miễn phí Kaggle: ~30 giờ/tuần, mỗi session tối đa ~9-12 giờ — đủ cho 1 lần chạy full pipeline. Sau khi xong, tải `results/` và `adapters/correct/` về (hoặc Save Version -> Save & Run All để giữ output).

In [ ]:
# Setup (chạy ô này trước)
# Kaggle cấp GPU T4 x2 theo mặc định; lab này không cần multi-GPU (HARDWARE-GUIDE.md),
# nên khoá về 1 GPU TRƯỚC KHI import torch, tránh device_map="auto" tự chia model
# ra cả hai card.
import os, subprocess, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

REPO = "https://github.com/lenomi09/Day21-Track3-2A202601228-LeNgocMinh.git"
if not os.path.exists("Day21-Track3-2A202601228-LeNgocMinh"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("Day21-Track3-2A202601228-LeNgocMinh")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list — same one-source-of-truth reason
# as the Colab bootstrap (see scripts/build_colab.py). torch is preinstalled on
# Kaggle and requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("commit :", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                  capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — bật Settings > Accelerator > GPU T4 x2")
print("visible GPUs:", torch.cuda.device_count(), "(phải là 1)")


In [ ]:
# 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke

In [ ]:
# 3. Core pipeline — NB1 -> NB5
# EVAL_LIMIT rút ngắn cả hai tập eval: để trống = full run (nộp được),
# 8 = smoke pass nhanh. STAGES cho phép resume sau khi 1 stage fail.
import os

COMPUTE_TIER = "T4"                    # CPU | LAPTOP | T4 | BIGGPU
EVAL_LIMIT   = ""                      # "" | "4" | "8" | "16" | "25"
STAGES       = "nb1 nb2 nb3 nb4 nb5"

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}

In [ ]:
# 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null